In [91]:
import os, pickle, socket, struct, threading, time
import pygame
import random


from netcode import NetClient

pygame.init()


(5, 0)

In [92]:
screen = pygame.display.set_mode((1920, 1080))
backgroundImage = pygame.image.load('redSky.jpg').convert()
backgroundImage = pygame.transform.scale(backgroundImage, (1920, 1080))
backgroundImage2 = pygame.image.load('blueSky.jpg').convert()
backgroundImage2 = pygame.transform.scale(backgroundImage2, (1920, 1080))
winImage = pygame.image.load('funny_victory.jpg').convert()
winImage = pygame.transform.scale(winImage, (1920, 1080))
terrain  = pygame.image.load(("terrainSheet.png")).convert_alpha()
platform = terrain.subsurface(pygame.Rect(272, 16, 48, 16))
platform = pygame.transform.scale_by(platform, 3)
player_image = pygame.image.load("frog.png").convert_alpha()
player_image = pygame.transform.scale_by(player_image, 3)
world_tileset = pygame.image.load("world_tileset.png").convert_alpha()
ground_tile = world_tileset.subsurface(pygame.Rect(0, 0, 16, 16))
ground_tile = pygame.transform.scale_by(ground_tile, 3)
below_tile = world_tileset.subsurface(pygame.Rect(0, 16, 16, 16))
below_tile = pygame.transform.scale_by(below_tile, 3)
pumpkin = world_tileset.subsurface(pygame.Rect(64, 128, 16, 16))
pumpkin = pygame.transform.scale_by(pumpkin, 3)
bush = world_tileset.subsurface(pygame.Rect(16, 48, 16, 16))
bush = pygame.transform.scale_by(bush, 3)
exclamation_mark = world_tileset.subsurface(pygame.Rect(16, 32, 16, 16))
exclamation_mark = pygame.transform.scale_by(exclamation_mark, 3)





In [93]:
PLATFORM_WIDTH, PLATFORM_HEIGHT = platform.get_size()

PLATFORMS = [
    (480, 872),
    (1216, -1160),
    (984, -1656),
    (484, -3356),
    (352, -4592),
    (1060, -4952),
    (484, -5848),
    (840, -6032),
    (1184, -6208),
    (256, -6932),
    (896, -7308),
    (856, -8068),
    (200, -8452),
    (488, -8644),
    (1184, -9016),
    (972, -9804), 
]


In [94]:
GROUND_WIDTH, GROUND_HEIGHT = ground_tile.get_size()

GROUND = [
    (x, 1080 - GROUND_HEIGHT) for x in range(-100, 2220, GROUND_WIDTH)
]

In [95]:
BELOW_WIDTH, BELOW_HEIGHT = below_tile.get_size()
BELOW_ROWS = 10

BELOW = [
    (x, 1080 - BELOW_HEIGHT + 48 + row * BELOW_HEIGHT)
    for row in range(BELOW_ROWS)
    for x in range(0, 1920, BELOW_WIDTH)
]

In [96]:
def make_island(x, y, width, height):
    assert 2 <= width <= 18, "island width should be 2-18 blocks"
    assert 2 <= height <= 10, "island height should be 2-10 blocks"

    ground_tiles = [(x + col * GROUND_WIDTH, y) for col in range(width)]

    below_tiles = [
        (x + col * BELOW_WIDTH, y + row * BELOW_HEIGHT)
        for row in range(1, height)
        for col in range(width)
    ]

    return ground_tiles, below_tiles

In [97]:
ISLANDS = [
    (720, 724, 13, 3),
    (1452, 564, 9, 3),
    (872, 416, 7, 3),
    (420, 256, 8, 3),
    (1084, 104, 4, 3),
    (572, -52, 5, 3),
    (1176, -208, 15, 3),
    (6, -368, 15, 3),
    (1484, -458, 5, 2),
    (776, -684, 13, 3),
    (900, -882, 8, 2),
    (1564, -992, 6, 3),
    (580, -1320, 11, 3),
    (1296, -1488, 11, 3),
    (468, -1820, 9, 3),
    (1060, -1992, 8, 3),
    (500, -2152, 8, 3),
    (76, -2324, 6, 3),
    (524, -2492, 11, 3),
    (1236, -2660, 11, 3),
    (560, -2824, 10, 3),
    (1236, -3004, 9, 3),
    (812, -3184, 5, 3),
    (44, -3536, 6, 3),
    (472, -3716, 6, 3),
    (856, -3884, 8, 3),
    (1424, -4064, 6, 3),
    (936, -4240, 6, 3),
    (600, -4416, 5, 3),
    (704, -4772, 5, 3),
    (1352, -5128, 7, 3),
    (956, -5304, 4, 3),
    (508, -5492, 7, 3),
    (84, -5672, 5, 3),
    (1456, -6388, 4, 3),
    (956, -6564, 7, 3),
    (552, -6748, 5, 3),
    (568, -7116, 4, 3),
    (1216, -7496, 3, 3),
    (1584, -7688, 5, 3),
    (1164, -7872, 4, 3),
    (484, -8260, 3, 3),
    (764, -8828, 4, 3),
    (1552, -9208, 3, 3),
    (1156, -9408, 4, 3),
    (656, -9604, 5, 3),
    (1200, -10000, 15, 5),
]

for ix, iy, iw, ih in ISLANDS:
    island_ground, island_below = make_island(ix, iy, iw, ih)
    GROUND += island_ground
    BELOW += island_below

In [98]:
PUMPKIN_WIDTH, PUMPKIN_HEIGHT = pumpkin.get_size()

def make_pumpkins(islands, seed=1334):
    rng = random.Random(seed)
    ground_y = 1080 - GROUND_HEIGHT - PUMPKIN_HEIGHT
    pumpkins = []

    x = 500
    pumpkins.append((x, ground_y))
        

    island_index = rng.randint(3, 6)
    while island_index < len(islands):
        ix, iy, iwidth, _ = islands[island_index]
        px = ix + rng.randint(0, iwidth - 1) * GROUND_WIDTH
        py = iy - PUMPKIN_HEIGHT
        pumpkins.append((px, py))
        island_index += rng.randint(3, 6)

    return pumpkins

PUMPKINS = make_pumpkins(ISLANDS)

In [99]:
BUSH_WIDTH, BUSH_HEIGHT = bush.get_size()

def make_bushes(islands, seed=1334):
    rng = random.Random(seed)
    ground_y = 1080 - GROUND_HEIGHT - BUSH_HEIGHT
    bushes = []

    x = 1000
    bushes.append((x, ground_y))
        

    island_index = rng.randint(1, 2)
    while island_index < len(islands):
        ix, iy, iwidth, _ = islands[island_index]
        px = ix + rng.randint(0, iwidth - 1) * GROUND_WIDTH
        py = iy - BUSH_HEIGHT
        bushes.append((px, py))
        island_index += rng.randint(1, 2)

    return bushes

BUSHES = make_bushes(ISLANDS)

In [100]:

#Temporary function that will be replaced by the sprite function
_tint_cache = {}

def tinted_frog(tint):
    key = tuple(tint)
    if key not in _tint_cache:
        img = player_image.copy()
        img.fill((tint[0], tint[1], tint[2], 255),
                 special_flags=pygame.BLEND_RGBA_MULT)
        _tint_cache[key] = img
    return _tint_cache[key]


In [101]:
class Camera:

    def __init__(self):
        self.offset_x = 0
        self.offset_y = 0
        self.move_speed = 5
        self.image = backgroundImage

    def follow(self, player):
        #self.offset_x = screen.get_width() // 2 - player.x
        self.offset_y = screen.get_height() // 2 - player.y

    def render_world(self, platforms, ground, below, pumpkins,bushes,  remotes=()):
        self.follow(player)

        screen.fill((0, 0, 0))
        
        if (player.y <= -5000):
            self.image = backgroundImage2
        elif(player.y > -5000):
            self.image = backgroundImage


        if(player.x >= 1600 and player.y <= -10000):
            self.image = winImage
        
        screen.blit(self.image, (0, 0))

        screen.blit(exclamation_mark, (1600+self.offset_x, -10100+self.offset_y))

        for x, y in platforms:
            screen.blit(platform, (x+self.offset_x, y+self.offset_y))

        for x, y in ground:
            screen.blit(ground_tile, (x+self.offset_x, y+self.offset_y))
        
        for x, y in below:
            screen.blit(below_tile, (x+self.offset_x, y+self.offset_y))

        for x, y in pumpkins:
            screen.blit(pumpkin, (x+self.offset_x, y+self.offset_y))

        for x, y in bushes:
            screen.blit(bush, (x+self.offset_x, y+self.offset_y))

       
        for p in remotes:
            screen.blit(tinted_frog(p.tint), (p.x + self.offset_x, p.y + self.offset_y))

        screen.blit(player.image, (player.x + self.offset_x, player.y + self.offset_y))

In [102]:
class Player:
  def __init__(self, image, camera: Camera):
      #Koordinaterna
      self.x = 300
      self.y = 300
      self.speed = 10
      self.vel_y = 0
      self.gravity = 0.75
      self.jump_power = -20
      self.on_ground = False
      self.image = image
      self.camera = camera
      self.rect = pygame.Rect(self.x, self.y, image.get_width(), image.get_height())

  def get_rect(self):
      return pygame.Rect(self.x, self.y, self.image.get_width(), self.image.get_height())

  def move(self, left, right, jump):
    #Writen here due to a bug that has appeared that might have to do with how pygame is built
    self.speed = 10
    
    if left:
      self.x -= self.speed

    if right:
      self.x += self.speed

    if jump and self.on_ground:
        self.vel_y = self.jump_power
        self.on_ground = False

  def apply_gravity(self):
      self.vel_y += self.gravity
      if self.vel_y >= 20:
          self.vel_y = 20

      self.y += self.vel_y

In [103]:
def handle_input():
  keys = pygame.key.get_pressed()

  left = keys[pygame.K_a]
  right = keys[pygame.K_d]
  jump = keys[pygame.K_SPACE]

  return left, right, jump

In [104]:
SOLID_RECTS = (
    [pygame.Rect(x, y, PLATFORM_WIDTH, PLATFORM_HEIGHT) for x, y in PLATFORMS]
    + [pygame.Rect(x, y, GROUND_WIDTH, GROUND_HEIGHT) for x, y in GROUND]
    + [pygame.Rect(x, y, BELOW_WIDTH, BELOW_HEIGHT) for x, y in BELOW]
)

In [105]:
def handle_platform_collision(player, platform_rects):
    player.rect = player.get_rect()
    player.on_ground = False

    for platform_rect in platform_rects:
        if not player.rect.colliderect(platform_rect):
            continue

        overlap_top = player.rect.bottom - platform_rect.top
        overlap_bottom = platform_rect.bottom - player.rect.top
        overlap_left = player.rect.right - platform_rect.left
        overlap_right = platform_rect.right - player.rect.left

        if player.vel_y > 0 and overlap_top <= overlap_left and overlap_top <= overlap_right:
            player.y -= overlap_top
            player.vel_y = 0
            player.on_ground = True
        elif player.vel_y < 0 and overlap_bottom <= overlap_left and overlap_bottom <= overlap_right:
            player.y += overlap_bottom
            player.vel_y = 0
        elif overlap_left <= overlap_right:
            player.x -= overlap_left
        else:
            player.x += overlap_right

        player.rect = player.get_rect()

    return player.on_ground

In [106]:
"""GROUND_TOLERANCE = 12

def is_Grounded(player: Player):
    player_left = player.x
    player_right = player.x + player.image.get_width()
    player_bottom = player.y + player.image.get_height()

    for x, y in PLATFORMS:
        landed_on_top = abs(player_bottom - y) <= GROUND_TOLERANCE
        within_platform_x = player_right > x and player_left < x + PLATFORM_WIDTH
        if landed_on_top and within_platform_x:
            return True

    for x, y in GROUND:
        landed_on_top = abs(player_bottom - y) <= GROUND_TOLERANCE
        within_ground_x = player_right > x and player_left < x + GROUND_WIDTH
        if landed_on_top and within_ground_x:
            return True

    return False
    """

'GROUND_TOLERANCE = 12\n\ndef is_Grounded(player: Player):\n    player_left = player.x\n    player_right = player.x + player.image.get_width()\n    player_bottom = player.y + player.image.get_height()\n\n    for x, y in PLATFORMS:\n        landed_on_top = abs(player_bottom - y) <= GROUND_TOLERANCE\n        within_platform_x = player_right > x and player_left < x + PLATFORM_WIDTH\n        if landed_on_top and within_platform_x:\n            return True\n\n    for x, y in GROUND:\n        landed_on_top = abs(player_bottom - y) <= GROUND_TOLERANCE\n        within_ground_x = player_right > x and player_left < x + GROUND_WIDTH\n        if landed_on_top and within_ground_x:\n            return True\n\n    return False\n    '

In [107]:
PLAYER_NAME = "david"
SERVER_HOST = "127.0.0.1"

try:
    net.close()
except NameError:
    pass

net = NetClient(SERVER_HOST, PLAYER_NAME)
print("connected as player", net.my_id, "| tint", net.tint, "| resume", net.resume)


connected as player 6 | tint (255, 255, 255) | resume (1684, -10096.5)


In [108]:
camera = Camera()
player = Player(player_image, camera)

if net.resume is not None:
    player.x, player.y = net.resume

clock = pygame.time.Clock()


running = True
while running:
    for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
    
    left, right, jump = handle_input()
    
    player.move(left,right,jump)
    player.apply_gravity()
    handle_platform_collision(player, SOLID_RECTS)

    
    remotes = net.update(player.x, player.y)

    camera.render_world(PLATFORMS, GROUND, BELOW, PUMPKINS, BUSHES, remotes)

    status = "online" if net.connected else "OFFLINE"
    pygame.display.set_caption(
        f"{clock.get_fps():.0f} FPS | {PLAYER_NAME} #{net.my_id} | "
        f"{status} | {len(remotes)} others")
    pygame.display.flip()
    clock.tick(60)

net.close()
pygame.quit()